In [ ]:
%pip install monai nibabel tqdm scipy scikit-learn nbformat -q
%pip install --no-deps -q git+https://github.com/MIC-DKFZ/MedNeXt.git

In [ ]:
import glob
import os
import shutil
import urllib.request
import zipfile
from copy import deepcopy

import nibabel as nib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.data import CacheDataset, DataLoader
from monai.inferers import sliding_window_inference
from monai.transforms import (
    Compose,
    ConvertToMultiChannelBasedOnBratsClassesd,
    CropForegroundd,
    EnsureChannelFirstd,
    LoadImaged,
    NormalizeIntensityd,
    Orientationd,
    RandFlipd,
    RandScaleIntensityd,
    RandShiftIntensityd,
    RandSpatialCropd,
)
from sklearn.model_selection import train_test_split
from torch.cuda.amp import GradScaler, autocast
from tqdm.auto import tqdm

In [ ]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def dice_score(a, b):
    inter = float(((a > 0) & (b > 0)).sum())
    denom = float((a > 0).sum() + (b > 0).sum())
    return 2 * inter / denom if denom else 1.0


class DiceAndCE(nn.Module):
    def __init__(self, n_classes=3, smooth=1.0):
        super().__init__()
        self.n_classes = n_classes
        self.smooth = smooth

    def forward(self, logits, target):
        target_idx = target.argmax(dim=1)
        ce = F.cross_entropy(logits, target_idx)
        prob = F.softmax(logits, dim=1)
        t_one = target
        num = 2.0 * (prob * t_one).sum(dim=(2, 3, 4)) + self.smooth
        den = prob.sum(dim=(2, 3, 4)) + t_one.sum(dim=(2, 3, 4)) + self.smooth
        dice = 1.0 - (num / den).mean()
        return ce + dice

In [ ]:
from nnunet_mednext.network_architecture.mednextv1.MedNextV1 import MedNeXt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MedNeXt(
    in_channels=4,
    n_channels=32,
    n_classes=3,
    exp_r=[2, 3, 4, 4, 4, 4, 4, 3, 2],
    kernel_size=5,
    deep_supervision=False,
    do_res=True,
    do_res_up_down=True,
    block_counts=[2, 2, 2, 2, 2, 2, 2, 2, 2],
    norm_type="group",
    dim="3d",
    grn=True,
).to(device)
print("params:", count_parameters(model))

In [ ]:
DATA_ROOT = "data/BraTS2021_Training_Data"
ZIP_URL = (
    "https://zenodo.org/records/19541844/files/BraTS2021_Training_Data.zip?download=1"
)
if not glob.glob(os.path.join(DATA_ROOT, "BraTS*")):
    parent = os.path.dirname(DATA_ROOT)
    os.makedirs(parent, exist_ok=True)
    zip_path = os.path.join(parent, "BraTS2021_Training_Data.zip")
    urllib.request.urlretrieve(ZIP_URL, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(parent)
    os.remove(zip_path)
    print("downloaded and extracted BraTS2021_Training_Data")

OUTPUT_DIR = "mednext_xs_output"
SEED = 0
VALID_FRAC = 0.2
PATCH_SIZE = 96
BATCH_SIZE = 2
EPOCHS = 150
LR = 1e-4
WEIGHT_DECAY = 3e-5
NUM_WORKERS = 4
INFER_OVERLAP = 0.5

LABELS = [1, 2, 4]
MODALITIES = ["flair", "t1", "t1ce", "t2"]

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

case_dirs = sorted(glob.glob(os.path.join(DATA_ROOT, "BraTS*")))
train_dirs, val_dirs = train_test_split(
    case_dirs, test_size=VALID_FRAC, random_state=SEED
)
print("train cases:", len(train_dirs), "| val cases:", len(val_dirs))


def make_case_dict(case_dir):
    base = os.path.basename(case_dir)
    return {
        "image": [os.path.join(case_dir, f"{base}_{m}.nii.gz") for m in MODALITIES],
        "label": os.path.join(case_dir, f"{base}_seg.nii.gz"),
    }


train_files = [make_case_dict(d) for d in train_dirs]
val_files = [make_case_dict(d) for d in val_dirs]

train_transforms = Compose(
    [
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image"]),
        ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
        CropForegroundd(keys=["image", "label"], source_key="image"),
        RandSpatialCropd(
            keys=["image", "label"],
            roi_size=(PATCH_SIZE, PATCH_SIZE, PATCH_SIZE),
            random_size=False,
        ),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
        RandScaleIntensityd(keys="image", factors=0.1, prob=1.0),
        RandShiftIntensityd(keys="image", offsets=0.1, prob=1.0),
    ]
)

val_transforms = Compose(
    [
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image"]),
        ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    ]
)

train_ds = CacheDataset(
    data=train_files, transform=train_transforms, cache_num=len(train_files)
)
val_ds = CacheDataset(
    data=val_files, transform=val_transforms, cache_num=len(val_files)
)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

In [ ]:
model = MedNeXt(in_channels=4, n_channels=32, n_classes=3, kernel_size=5, grn=True).to(
    device
)
print("params:", count_parameters(model))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = DiceAndCE(n_classes=3)
scaler = GradScaler()
best_val = 0.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    steps = 0
    pbar = tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}", leave=False)
    for batch in pbar:
        image = batch["image"].to(device)
        label = batch["label"].to(device)
        optimizer.zero_grad()
        with autocast():
            logits = model(image)
            loss = criterion(logits, label)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += float(loss.item())
        steps += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    train_loss /= max(steps, 1)

    model.eval()
    val_loss = 0.0
    val_steps = 0
    with torch.no_grad():
        for batch in val_loader:
            image = batch["image"].to(device)
            label = batch["label"].to(device)
            with autocast():
                logits = model(image)
                loss = criterion(logits, label)
            val_loss += float(loss.item())
            val_steps += 1
    val_loss /= max(val_steps, 1)

    if best_val == 0.0 or val_loss < best_val:
        best_val = val_loss
        best_state = deepcopy(model.state_dict())
        torch.save(best_state, os.path.join(OUTPUT_DIR, "mednext_xs_best.pth"))
    print(
        f"epoch {epoch}: train={train_loss:.4f} val={val_loss:.4f} best={best_val:.4f}"
    )

torch.save(best_state, os.path.join(OUTPUT_DIR, "mednext_xs_best.pth"))
print("training done, best val_loss:", round(best_val, 4))

In [ ]:
model.load_state_dict(
    torch.load(os.path.join(OUTPUT_DIR, "mednext_xs_best.pth"), map_location=device)
)
model.eval()


def predict_volume(
    model, case_dir, device, patch_size=PATCH_SIZE, overlap=INFER_OVERLAP
):
    base = os.path.basename(case_dir)
    images = []
    affine = None
    for m in MODALITIES:
        img = nib.load(os.path.join(case_dir, f"{base}_{m}.nii.gz"))
        images.append(img.get_fdata(dtype=np.float32))
        affine = img.affine
    volume = np.stack(images, axis=0)
    for c in range(volume.shape[0]):
        vals = volume[c][volume[c] > 0]
        if vals.size:
            mean, std = float(vals.mean()), float(vals.std())
            if std > 0:
                volume[c] = (volume[c] - mean) / std
    tensor = torch.from_numpy(volume).unsqueeze(0).to(device)
    with torch.no_grad():
        with autocast():
            logits = sliding_window_inference(
                tensor,
                roi_size=(patch_size, patch_size, patch_size),
                sw_batch_size=2,
                predictor=model,
                overlap=overlap,
            )
    idx = logits[0].argmax(dim=0).detach().cpu().numpy()
    mask = np.zeros(idx.shape, dtype=np.uint8)
    for i, lab in enumerate(LABELS):
        mask[idx == i] = lab
    return mask, affine


val_cases = []
for case_dir in val_dirs:
    gt_path = os.path.join(case_dir, os.path.basename(case_dir) + "_seg.nii.gz")
    gt = nib.load(gt_path).get_fdata().astype(np.uint8)
    pred, affine = predict_volume(model, case_dir, device)
    pred = np.clip(pred, 0, 255).astype(np.uint8)
    if pred.shape != gt.shape:
        continue
    val_cases.append({"case": os.path.basename(case_dir), "pred": pred, "gt": gt})

ncr_dice = np.mean([dice_score(c["pred"] == 1, c["gt"] == 1) for c in val_cases])
ed_dice = np.mean([dice_score(c["pred"] == 2, c["gt"] == 2) for c in val_cases])
et_dice = np.mean([dice_score(c["pred"] == 4, c["gt"] == 4) for c in val_cases])
whole_dice = np.mean([dice_score(c["pred"], c["gt"]) for c in val_cases])
print(
    "val dice  NCR:",
    round(ncr_dice, 4),
    " ED:",
    round(ed_dice, 4),
    " ET:",
    round(et_dice, 4),
    " whole:",
    round(whole_dice, 4),
)

for c in val_cases[:3]:
    out_path = os.path.join(OUTPUT_DIR, f"{c['case']}_pred.nii.gz")
    nib.save(nib.Nifti1Image(c["pred"], np.eye(4)), out_path)
    print("saved:", out_path)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/mednext_xs_brats"
os.makedirs(DRIVE_DIR, exist_ok=True)
shutil.copy(
    os.path.join(OUTPUT_DIR, "mednext_xs_best.pth"),
    os.path.join(DRIVE_DIR, "mednext_xs_best.pth"),
)
print("saved weights to", os.path.join(DRIVE_DIR, "mednext_xs_best.pth"))